In [0]:
df = spark.read.table("workspace.bronze.erp_cust_az12")

In [0]:
df.display()

### Data Transformations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# 1. Define your mapping
mapping = {
    'CID': 'customer_key',
    'BDATE': 'birth_date',
    'GEN': 'gender'
}

# 2. Apply all transformations in one select statement
# This trims all strings, removes "NAS" from CID, and renames columns
df1 = df.select([
    (
        F.regexp_replace(F.trim(F.col(c)), "^...", "") if c == 'CID' 
        else F.trim(F.col(c)) if isinstance(df.schema[c].dataType, StringType) 
        else F.col(c)
    ).alias(mapping.get(c, c)) 
    for c in df.columns
])

display(df1)

In [0]:
## check for duplicates, if no duplicate values then write to delta table
duplicate_count = df1.groupBy(df1.columns[0]).count().filter("count > 1")
display(duplicate_count)

## Write to Silver table

In [0]:
df1.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.erp_customer_info_cleaned")